#Additional task: Fine-tune T5Gemma-2-270M on SQuAD and run Exp 1, 3, 4


In [1]:
!pip install -q transformers datasets evaluate bert-score accelerate sentencepiece


    extract-msg (<=0.29.*)
                 ~~~~~~~^

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [3]:
from pathlib import Path
import re
from collections import Counter
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from bert_score import score as bert_score


/Users/maksimpiskaev/Проекты/IR/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Paths for Kaggle
BASE_INPUT = Path("/kaggle/input/datasets/theycallmemax/content-2")
WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

BASE_SAMPLE_PATH = BASE_INPUT / 'task5_base_sample1000.parquet'
TOP1_CE_PATH = BASE_INPUT / 'task4_top1_ce_k100_fixed1000.parquet'
TOP1_DENSE_PATH = BASE_INPUT / 'task4_top1_dens e_minilm_sample1000.parquet'

print(BASE_SAMPLE_PATH.exists(), TOP1_CE_PATH.exists(), TOP1_DENSE_PATH.exists())


True True True


In [6]:
MODEL_ID = 'google/t5gemma-2-270m-270m'
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 32
BERTSCORE_MODEL = 'distilroberta-base'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


device: cpu


## Fine-tune on SQuAD


In [17]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)


Loading weights:   0%|          | 0/911 [00:00<?, ?it/s]

In [18]:
squad = load_dataset('squad')
train_ds = squad['train'].shuffle(seed=42).select(range(40000))
val_ds = squad['validation'].shuffle(seed=42).select(range(4000))


def preprocess_squad(batch):
    inputs = [f"question: {q} context: {c}" for q, c in zip(batch["question"], batch["context"])]
    targets = [a["text"][0] if len(a["text"]) else "" for a in batch["answers"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess_squad, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess_squad, batched=True, remove_columns=val_ds.column_names)


In [ ]:
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,                 # важно
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

args = Seq2SeqTrainingArguments(
    output_dir=str(WORK_DIR / "t5gemma_squad_ft"),
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=2,
    predict_with_generate=True,
    logging_steps=100,
    fp16=False,          # важно: выключить
    bf16=use_bf16,       # только если реально поддерживается
    max_grad_norm=0.0,   # обход клиппинга, который триггерит unscale
    report_to=[],
)


trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    processing_class=tokenizer,  
)


trainer.train()
trainer.save_model(str(WORK_DIR / 't5gemma_squad_ft_final'))


Step,Training Loss,Validation Loss
1000,0.409267,0.433991
2000,0.301473,0.345650


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path
import shutil

EXPORT_DIR = Path(WORK_DIR) / "t5gemma_squad_ft_final"
ARCHIVE_BASE = str(Path(WORK_DIR) / "t5gemma_squad_ft_final")

print("Export dir exists:", EXPORT_DIR.exists())
print("Files:", [p.name for p in EXPORT_DIR.iterdir()])

shutil.make_archive(ARCHIVE_BASE, "zip", str(EXPORT_DIR))
print("Archive ready:", ARCHIVE_BASE + ".zip")


In [7]:
# Reload fine-tuned model (local path)
FT_MODEL_PATH = Path("/Users/maksimpiskaev/Проекты/IR/HW_5/models/t5gemma_squad_ft_final")


print("exists:", Path(FT_MODEL_PATH).exists())
print("files:", [p.name for p in Path(FT_MODEL_PATH).iterdir()][:10])

ft_tokenizer = AutoTokenizer.from_pretrained(str(FT_MODEL_PATH), local_files_only=True)
ft_model = AutoModelForSeq2SeqLM.from_pretrained(str(FT_MODEL_PATH), local_files_only=True).to(DEVICE)
ft_model.eval()
print("Reload OK")


exists: True
files: ['model.safetensors', 'tokenizer_config.json', 'config.json', 'tokenizer.json', 'generation_config.json', 'training_args.bin']


Loading weights: 100%|██████████| 911/911 [00:00<00:00, 7209.73it/s]

Reload OK


## Metrics helpers


In [8]:
_ws_re = re.compile(r'\s+')
_punct_re = re.compile(r'[^\w\s]')

def normalize_text(text: str) -> str:
    s = str(text).lower()
    s = _punct_re.sub(' ', s)
    s = _ws_re.sub(' ', s).strip()
    return s

def to_answers(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [str(a) for a in x if str(a).strip()]
    s = str(x).strip()
    return [s] if s else []

def em_squad(pred: str, refs: list[str]) -> float:
    p = normalize_text(pred)
    return float(any(p == normalize_text(r) for r in refs))

def f1_pair(pred: str, ref: str) -> float:
    p_tokens = normalize_text(pred).split()
    r_tokens = normalize_text(ref).split()
    if not p_tokens and not r_tokens:
        return 1.0
    if not p_tokens or not r_tokens:
        return 0.0
    common = Counter(p_tokens) & Counter(r_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p_tokens)
    recall = overlap / len(r_tokens)
    return 2 * precision * recall / (precision + recall)

def f1_squad(pred: str, refs: list[str]) -> float:
    return max((f1_pair(pred, r) for r in refs), default=0.0)

def em_loose_mirage(pred: str, refs: list[str]) -> float:
    p = normalize_text(pred)
    if not p:
        return 0.0
    for r in refs:
        rr = normalize_text(r)
        if rr and (p in rr or rr in p):
            return 1.0
    return 0.0

def em_strict_mirage(pred: str, refs: list[str]) -> float:
    return em_squad(pred, refs)


## Inference helper


In [9]:
def t5_answer(question: str, context: str = "") -> str:
    prompt = f"question: {question} context: {context}" if context else f"question: {question}"

    inputs = ft_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_TARGET_LEN,
            # явно задаём только валидные для greedy-пути аргументы
            num_beams=1,
        )

    ans = ft_tokenizer.decode(out[0], skip_special_tokens=True).strip()
    ans = ans.split("\\n")[0].strip()
    if "." in ans:
        ans = ans.split(".")[0].strip()
    return ans


In [10]:
base_df = pd.read_parquet(BASE_SAMPLE_PATH).copy()
base_df['query_id'] = base_df['query_id'].astype(str)
base_df['answers'] = base_df['answers'].apply(to_answers)
base_df.head(2)


,query_id,question,answers,oracle
0,8830b86a-46bf-467d-8cd3-be7fe70fc28d,the school of gymnastics is called in athens as,"[the Academy, the Lyceum, the Cynosarges]","{'doc_chunk': 'Accordingly, the gymnasium beca..."
1,57465897-9a01-4d92-8738-1fa713da6df0,where did duke get the name blue devils,"[from the French ""les Diables Bleus"" or ""the B...",{'doc_chunk': 'Intercollegiate sports teams of...


## Experiment 1: no context (0-shot style)


In [42]:
print("DEVICE:", DEVICE)
print("model device:", next(ft_model.parameters()).device)

DEVICE: cuda
model device: cuda:0


In [12]:
from tqdm.auto import tqdm

def run_eval(df_eval, context_col=None, exp_name='exp'):
    rows = []
    for r in tqdm(df_eval.itertuples(index=False), total=len(df_eval), desc=exp_name):
        ctx = '' if context_col is None else getattr(r, context_col)
        pred = t5_answer(r.question, ctx)
        rows.append({
            'query_id': r.query_id,
            'question': r.question,
            'answers': r.answers,
            'prediction': pred
        })
    pred_df = pd.DataFrame(rows)

    em_vals, f1_vals, loose_vals, strict_vals = [], [], [], []
    best_refs = []
    for r in pred_df.itertuples(index=False):
        refs = r.answers if isinstance(r.answers, list) else []
        em_vals.append(em_squad(r.prediction, refs))
        f1_vals.append(f1_squad(r.prediction, refs))
        loose_vals.append(em_loose_mirage(r.prediction, refs))
        strict_vals.append(em_strict_mirage(r.prediction, refs))
        best_refs.append(max(refs, key=lambda x: f1_pair(r.prediction, x)) if refs else '')

    pred_texts = pred_df['prediction'].astype(str).str.strip().replace('', '[NO_ANSWER]').tolist()
    ref_texts = [str(x).strip() if str(x).strip() else '[NO_ANSWER]' for x in best_refs]
    _P, _R, F = bert_score(pred_texts, ref_texts, model_type=BERTSCORE_MODEL, lang='en', verbose=False)

    metrics = pd.DataFrame([{
        'experiment': exp_name,
        'EM': float(np.mean(em_vals)),
        'F1': float(np.mean(f1_vals)),
        'EM_loose': float(np.mean(loose_vals)),
        'EM_strict': float(np.mean(strict_vals)),
        'BERTScore_F1': float(F.mean().item()),
        'N': int(len(pred_df))
    }])
    return pred_df, metrics


In [46]:
def t5_answer(question: str, context: str = "") -> str:
    prompt = f"question: {question} context: {context}" if context else f"question: {question}"

    inputs = ft_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LEN,
        )

    ans = ft_tokenizer.decode(out[0], skip_special_tokens=True).strip()
    ans = ans.split("\\n")[0].strip()
    if "." in ans:
        ans = ans.split(".")[0].strip()
    return ans

pred1, m1 = run_eval(base_df[['query_id','question','answers']].copy(), None, 'exp1_no_context')
m1


[exp1_no_context] rows=1000, model_device=cuda:0


exp1_no_context:   0%|          | 0/1000 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,experiment,EM,F1,EM_loose,EM_strict,BERTScore_F1,N
0,exp1_no_context,0.002,0.030985,0.017,0.002,0.841883,1000


## Experiment 3: oracle context


In [13]:
base_df['oracle_context'] = base_df['oracle'].apply(lambda o: str(o.get('doc_chunk','')) if isinstance(o, dict) else '')
pred3, m3 = run_eval(base_df[['query_id','question','answers','oracle_context']].copy(), 'oracle_context', 'exp3_oracle')
m3


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6903.05it/s]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,experiment,EM,F1,EM_loose,EM_strict,BERTScore_F1,N
0,exp3_oracle,0.183,0.405836,0.786,0.183,0.916063,1000


## Experiment 4: top-1 context from two rankers


In [14]:
ce_top1 = pd.read_parquet(TOP1_CE_PATH)[['query_id','passage']].rename(columns={'passage':'context_ce'})
dense_top1 = pd.read_parquet(TOP1_DENSE_PATH)[['query_id','passage']].rename(columns={'passage':'context_dense'})
ce_top1['query_id'] = ce_top1['query_id'].astype(str)
dense_top1['query_id'] = dense_top1['query_id'].astype(str)

exp4_ce_df = base_df[['query_id','question','answers']].merge(ce_top1, on='query_id', how='left').fillna({'context_ce':''})
exp4_dense_df = base_df[['query_id','question','answers']].merge(dense_top1, on='query_id', how='left').fillna({'context_dense':''})

pred4_ce, m4_ce = run_eval(exp4_ce_df, 'context_ce', 'exp4_top1_ce')
pred4_dense, m4_dense = run_eval(exp4_dense_df, 'context_dense', 'exp4_top1_dense')

pd.concat([m4_ce, m4_dense], ignore_index=True)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6008.28it/s]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6274.98it/s]
RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- U

,experiment,EM,F1,EM_loose,EM_strict,BERTScore_F1,N
0,exp4_top1_ce,0.095,0.241787,0.475,0.095,0.885344,1000
1,exp4_top1_dense,0.101,0.247022,0.465,0.101,0.886107,1000


## Общий вывод
- На `exp1_no_context` (без контекста) модель после fine-tune показывает низкие точные метрики: `EM=0.002`, `F1=0.030985`, `EM_loose=0.017`, `BERTScore_F1=0.841883` (`N=1000`).
- Добавление oracle-контекста (`exp3_oracle`) существенно улучшает качество: `EM=0.183`, `F1=0.405836`, `EM_loose=0.786`, `BERTScore_F1=0.916063` (`N=1000`).
- В `exp4` с retrieval top-1 контекстом результаты ниже oracle-режима: `exp4_top1_ce` — `EM=0.095`, `F1=0.241787`, `BERTScore_F1=0.885344`; `exp4_top1_dense` — `EM=0.101`, `F1=0.247022`, `BERTScore_F1=0.886107` (`N=1000`).
- Между двумя ranker'ами в `exp4` `Dense top-1` немного лучше `CE top-1` по `EM/F1/BERTScore`, но разница небольшая.
- Качество модели в первую очередь определяется качеством контекста; лучший режим в этих экспериментах — `oracle context`.
